# End-to-end simulation of observation and pipeline reduction of Barnard's star as a prevision RV standard

<span style="color:red">The HISPEC DRP is in active development. Please submit issues and suggestions on the [github](https://github.com/oirlab/HISPEC_DRP).</span>


This notebook shows two examples of simulating BSPEC data:

1. A simulation of a single up-the-ramp exposure of a star with a theoretical spectrum with similar properties to Barnard's star, assuming no planets.
2. A segment of a precision radial velocity observations of the same star, now with an orbiting planet causing the radial velocity to vary.

For each exposure, the notebook simulates a BSPEC rate map from a Phoenix template (approximately similar to Barnard's star), builds a `RampModel`, then runs `Stage0Pipeline` → `Stage1Pipeline` → `Stage2Pipeline`. An idealized etalon is also input on fiber 2, but is not used for wavelength determination.

Documentation relating to the data products produced by each pipeline stage can be found in the [API documentation](https://oirlab.github.io/HISPEC_DRP/api/generated/hispecdrp.datamodels.html). RSPEC data can be simulated by changing the SPECTROGRAPH keyword. In nominal instrument operation, both BSPEC and RSPEC exposures are obtained at the same time.

---

## Setup

**Environment variables** (set these in your shell / notebook kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache directory (and SQLite DB) |
| `KECK_OBSERVER_EMAIL` | Syncing / downloading from KOA | Keck Observer account email |
| `KECK_OBSERVER_PASSWORD` | Syncing / downloading from KOA | Keck Observer account password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.  
The KOA cell below syncs metadata from remote KOA and downloads any **BSPEC** calibration files that are not already in the cache.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/path/to/barnards_star_prv_example"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

# Requires env vars: KOA_CALIBRATION_CACHE, KECK_OBSERVER_EMAIL, KECK_OBSERVER_PASSWORD
SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Single exposure of star with no planet

### 2.1 Generate etalon spectrum

This etalon will be simulated as a custom line list on fiber 2. Currently, it is only used as an example and is not part of determining the wavelength solution, which is instead derived from the ZEMAX optical model.

In [ ]:
from hispecdrp.utils.spectral_templates_utils import etalon_line_list
import numpy as np
import astropy.units as u

etalon_order, etalon_wvs, etalon_finesse, etalon_trans = etalon_line_list(
    fsr=30 * u.GHz, # free spectral range
    wavelength_range=(950 * u.nm, 1550 * u.nm),
    n_cavity=1.0,
    wavelength_meas=np.array([1064, 1348, 1550]) * u.nm, # measured etalon data
    finesse_meas=np.array([39.5, 38.5, 37.3]),
    peak_trans_meas=np.array([0.584, 0.650, 0.654]),
)

etalon_flux = 1e4 *u.ph/u.s * etalon_trans

### 2.2 Simulated rate map from 1D spectra → `RampModel`

Load a Phoenix template, densify it (upsampling to sufficient resolution for HISPEC), Doppler-shift the densified 1D spectrum by
`radial_velocity_km_s` (`inject_rv_via='spectrum'`, default), paint with ZEMAX, then
build a raw UTR `RampModel`. 

1. Fiber 1 is the on the star assuming ideal coupling + sky background. 
2. Fiber 2 is an idealized unresolved etalon (generated in previous cell). 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind
- instrument thermal background

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum

FIBERS = [1,2,3]
READ_TIME = 1.7 # seconds


# Injected stellar RV (km/s). Default path densifies the 1D template, Doppler-shifts
# it, paints with ZEMAX, and attaches the fixed packaged line-ZP WAVE (no WAVE×(1+z)).
STELLAR_RV = -110.1

# WARNING: the memory required for ramp simulations scales with exposure time
# and will crash computer if requested memory exceeds what is available on your system
# e.g. a 60 second exposure requires 3 GB of memory, a 10 min exposure requires 30 GB
exposure_time = 60 # seconds
n_reads = int(exposure_time/READ_TIME)

obs_jd = 2461293.91667 # arbitrary observation date

# Create a UTR map from a controlled seed and template spectrum
# The user can normalize to a specific magnitude (AB mag) and 2MASS filter
ramp, cals, rate_map = create_ramp_from_spectrum(
    OUT,
    spectrograph=SPECTROGRAPH,
    fibers=FIBERS,
    template_filename={
        1:"lte03200-5.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5",
    },
    mag_ab=6.15,
    filter_band="J",  # BSPEC (yJ)
    fiber_spectra={
        2:(etalon_wvs,etalon_flux),
    },
    n_reads=n_reads,
    readtime=READ_TIME,
    seed=0,
    radial_velocity_km_s={
        1:STELLAR_RV,
    },
    barycentric_velocity_km_s=0.0,
    inject_rv_via="spectrum",
    sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
    apply_sky=[1,3],
    apply_e2e_throughput=[1,3],
    mjd_start=obs_jd-2400000.5,
    save_ramp=True,
    overwrite_ramp=True,
    poisson_noise=True,
    reuse_stage1_sims=True,
    register_stage1_sims=False
)

print(f"fibers: {FIBERS}")
print(f"ramp type: {ramp.meta.model_type}")
print(f"ramp filename: {ramp.meta.filename}")
print(f"ramp data shape: {ramp.data.shape}")
print(f"Stage-0 cals: {sorted(cals)}")
print(f"rate_map shape: {rate_map.shape}")
print(f"wavesol note: {getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}")


Add header keywords to the raw ramp

In [ ]:

# add fiber keywords
setattr(ramp.meta, 'object_fiber1', 'SCIENCE')
setattr(ramp.meta, 'object_fiber2', 'ETALON-B')
setattr(ramp.meta, 'object_fiber3', 'SKY')

# add exposure time keyword
setattr(ramp.meta, 'exposure_time', exposure_time)
setattr(ramp.meta, 'exposure_type', 'SPEC')


# set target keywords
setattr(ramp.meta, 'target_gaia_id', '4472832130942575872')
setattr(ramp.meta, 'target_name', 'Barnard Star')
setattr(ramp.meta, 'target_ra', 269.4520417)
setattr(ramp.meta, 'target_dec', 4.6933639)

# save again
ramp.save(filepath=os.path.join(OUT, 'HB.20260910.36000.29_raw.fits'))


### 2.2 Stage 0 → 1 → 2 via pipeline classes


In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline

os.environ["HISPEC_DRP_VIZIER_MIRRORS"] = "vizier.cfa.harvard.edu"

l0 = Stage0Pipeline(output_dir=OUT, dataset=ramp).apply()

do_fibers = [str(f) for f in FIBERS]
# Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
stage1_args = {
    "spec_extract": {"do_fibers": do_fibers},
    "set_wavesol": {"wavesol": cals["wavesol"]},
}
l1 = Stage1Pipeline(output_dir=OUT, primitives=stage1_args, dataset=l0).apply()

injected_m_s = float(STELLAR_RV) * 1000.0
l2 = Stage2Pipeline(
    output_dir=OUT,
    dataset=l1,
    primitives={
        "blaze_corr": {"do_fibers": do_fibers},
        "stellar_rv": {
            "run": False,
            "ccf_input_velocity": injected_m_s,
            "do_fibers": [1],
            "save_preview": True,
            "preview_write_png": True,
        },
        "sky_sub": {"run": True},
    },
).apply()

print(f"L0 type: {l0.meta.model_type}")
print(f"L0 filename: {l0.meta.filename}")
print(f"L1 type: {l1.meta.model_type}")
print(f"L1 filename: {l1.meta.filename}")
print(f"L2 type: {l2.meta.model_type}")
print(f"L2 filename: {l2.meta.filename}")

## 3. Segment of PRV observations

We now simulate multiple exposures over several nights for a super Earth planet orbiting the same Barnard's star analog.

### 3.1 Simulating rate maps from an initial 1D spectrum and varying RV

1. Fiber 1 is the on the star assuming ideal coupling + sky background. 
2. Fiber 2 is an idealized unresolved etalon (generated in previous cell). 
3. Fiber 3 is background sky flux.

What this simulation does **not** include:
- telluric injection/correction
- wavelength solution determination (here the wave solution is derived from a ZEMAX instrument model)
- atmospheric variability of any kind
- stellar variability of any kind

In [ ]:
import numpy as np


FIBERS = [1,2,3]
READ_TIME = 1.7 # seconds

# Injected stellar RV (km/s).
STELLAR_RV = -110.1

# observation parameters
starting_jd = 2461293.91667 # sep 10 2026
nights = 3
exposures_per_night = 3
exposure_time = 60 # seconds
n_reads = int(exposure_time/READ_TIME)

# creating array of observation start times
nightly_start_jd_arr = starting_jd + np.arange(0,nights)
time_offsets = np.linspace(0, (exposure_time+READ_TIME)/86400 * (exposures_per_night - 1), exposures_per_night)
obs_jd_arr = np.add.outer(nightly_start_jd_arr, time_offsets).flatten()

# planet parameters
phase0_jd = starting_jd # zero phase occurs when first observation starts
semiamplitude_K = 15 # m/s
period = 1.75 # days

# calculate RV at each observation
phase_arr = (obs_jd_arr - phase0_jd) / period
rv_arr = semiamplitude_K * np.sin(2 * np.pi * phase_arr)/1000 + STELLAR_RV


Define function to simulate and get L2 data product from each exposure

In [ ]:
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
import gc

def run_one_injection(label: str, injected_rv_km_s: float, mjd_start: float, seed: int) -> dict:
    """Build one UTR, reduce Stage0–2, return L2 data product."""
    run_dir = OUT
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")

    ramp, cals, rate_map = create_ramp_from_spectrum(
        OUT,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename={
            1:"lte03200-5.00-0.5.PHOENIX-ACES-AGSS-COND-2011-HiRe.h5",
        },
        mag_ab=6.15,
        filter_band="J",  # BSPEC (yJ)
        fiber_spectra={
            2:(etalon_wvs,etalon_flux),
        },
        n_reads=n_reads,
        readtime=READ_TIME,
        seed=seed,
        radial_velocity_km_s={
            1:STELLAR_RV,
        },
        barycentric_velocity_km_s=0.0,
        inject_rv_via="spectrum",
        sky_filename='mk_skybg_zm_1.0_1.0_ph.dat',
        apply_sky=[1,3],
        apply_e2e_throughput=[1,3],
        mjd_start=mjd_start-2400000.5,
        save_ramp=False,
        overwrite_ramp=True,
        poisson_noise=True,
        register_stage1_sims=False,
        register_wavesol=False,
    )

    # add fiber keywords
    setattr(ramp.meta, 'object_fiber1', 'SCIENCE')
    setattr(ramp.meta, 'object_fiber2', 'ETALON-B')
    setattr(ramp.meta, 'object_fiber3', 'SKY')

    # add exposure time keyword
    setattr(ramp.meta, 'exposure_time', exposure_time)
    setattr(ramp.meta, 'exposure_type', 'SPEC')

    # set target keywords
    setattr(ramp.meta, 'target_gaia_id', '4472832130942575872')
    setattr(ramp.meta, 'target_name', 'Barnard Star')
    setattr(ramp.meta, 'target_ra', 269.4520417)
    setattr(ramp.meta, 'target_dec', 4.6933639)
    
    wmeta = getattr(cals.get("wavesol"), "meta", None)
    print(
        f"ramp={ramp.meta.filename}  shape={ramp.data.shape}  "
        f"wavesol note={getattr(wmeta, 'note', None)}  "
        f"injected_rv_via={getattr(wmeta, 'injected_rv_via', None)}"
    )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=OUT, dataset=ramp).apply()

    del ramp
    del rate_map

    do_fibers = [str(f) for f in FIBERS]
    # Fixed packaged line-ZP WAVE from create_ramp_from_spectrum (unscaled for spectrum paint)
    stage1_args = {
        "spec_extract": {"do_fibers": do_fibers},
        "set_wavesol": {"wavesol": cals["wavesol"]},
    }
    l1 = Stage1Pipeline(output_dir=OUT, primitives=stage1_args, dataset=l0).apply()

    del l0

    injected_m_s = float(STELLAR_RV) * 1000.0
    l2 = Stage2Pipeline(
        output_dir=OUT,
        dataset=l1,
        primitives={
            "blaze_corr": {"do_fibers": do_fibers},
            "stellar_rv": {
                "run": False,
                "ccf_input_velocity": injected_m_s,
                "do_fibers": [1],
                "save_preview": True,
                "preview_write_png": True,
            },
            "sky_sub": {"run": True},
        },
    ).apply()

    del l1
    gc.collect()



Now run

In [ ]:
l2_results = {}

for i in range(nights*exposures_per_night):
    l2_results[str(obs_jd_arr[i])] = run_one_injection(label=str(obs_jd_arr[i]),
                                                       injected_rv_km_s=rv_arr[i],
                                                       mjd_start=obs_jd_arr[i],
                                                       seed=None)